In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# load data; filter by product and timeframe
file_names = [
	".\\r5_d2.csv",
	".\\r5_d3.csv",
	".\\r5_d4.csv",
]
merged_file_name = ".\\merged_r5.csv"

if not Path(merged_file_name).exists():
	parts = []
	for i, fn in enumerate(file_names):
		part = pd.read_csv(fn, delimiter=";")
		part["timestamp"] = part["timestamp"] + i * 1_000_000
		parts.append(part)

	pd.concat(parts, ignore_index=True).to_csv(merged_file_name, sep=";", index=False)

df = pd.read_csv(merged_file_name, delimiter=";")


low, high = map(int, input().split())
# low, high = (0, 3000000)
df = df[(df['timestamp'] >= low) & (df['timestamp'] <= high)]
df = df[(df['bid_price_1'].notna()) & (df['ask_price_1'].notna())]

df = df[df['product'] == 'GALAXY_SOUNDS_DARK_MATTER']

In [ ]:
import numpy as np


# plot 7 price levels
# plt.figure(figsize=(300, 30))
plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], df['bid_price_1'], label='bid_price_1', lw = 0.3)
plt.plot(df['timestamp'], df['bid_price_2'], label='bid_price_2', lw = 0.3)
plt.plot(df['timestamp'], df['bid_price_3'], label='bid_price_3', lw = 0.3)
plt.plot(df['timestamp'], df['mid_price'], label='mid_price', lw = 0.3)
plt.plot(df['timestamp'], df['ask_price_1'], label='ask_price_1', lw = 0.3)
plt.plot(df['timestamp'], df['ask_price_2'], label='ask_price_2', lw = 0.3)
plt.plot(df['timestamp'], df['ask_price_3'], label='ask_price_3', lw = 0.3)
x = df['timestamp'].to_numpy()
for y, c in [
	(df['bid_price_1'], 'C0'),
	(df['mid_price'], 'C3'),
	(df['ask_price_1'], 'C4'),
]:
	mask = y.notna()
	m, b = np.polyfit(x[mask], y[mask], 1)
	plt.plot(x, m * x + b, linestyle='--', linewidth=0.8, color=c)

plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.title('All Bid/Ask Levels + Mid Price')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

# lowest bid, mid price, highest ask
lowest_bid = df[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1)
highest_ask = df[['ask_price_1', 'ask_price_2', 'ask_price_3']].max(axis=1)
plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], lowest_bid, label='lowest_bid', lw = 0.3)
plt.plot(df['timestamp'], df['mid_price'], label='mid_price', lw = 0.3)
plt.plot(df['timestamp'], highest_ask, label='highest_ask', lw = 0.3)
plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.title('Lowest Bid, Mid Price, Highest Ask')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

# highest bid, mid price, lowest ask
highest_bid = df[['bid_price_1', 'bid_price_2', 'bid_price_3']].max(axis=1)
lowest_ask = df[['ask_price_1', 'ask_price_2', 'ask_price_3']].min(axis=1)
plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], highest_bid, label='highest_bid', lw = 0.3)
plt.plot(df['timestamp'], df['mid_price'], label='mid_price', lw = 0.3)
plt.plot(df['timestamp'], lowest_ask, label='lowest_ask', lw = 0.3)
plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.title('Highest Bid, Mid Price, Lowest Ask')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# plot differences in best bid + ask to find signal --> +/- 2.5!
best_bid = df[['bid_price_1', 'bid_price_2', 'bid_price_3']].max(axis=1)
best_ask = df[['ask_price_1', 'ask_price_2', 'ask_price_3']].min(axis=1)
best_bid_diff = best_bid.diff()
best_ask_diff = best_ask.diff()
plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], best_bid_diff, label='best_bid_diff', lw=1)
plt.plot(df['timestamp'], best_ask_diff, label='best_ask_diff', lw=1)
plt.xlabel('Timestamp')
plt.ylabel('Price Change')
plt.title('Consecutive Changes in Best Bid and Ask')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# best bid and ask
best_bid = df[['bid_price_1', 'bid_price_2', 'bid_price_3']].max(axis=1)
best_ask = df[['ask_price_1', 'ask_price_2', 'ask_price_3']].min(axis=1)
mid_price = (best_bid + best_ask) / 2

# rolling z-scores
window = 30  # adjust as needed
mid_mean = mid_price.rolling(window).mean()
mid_std = mid_price.rolling(window).std()
z_score = (mid_price - mid_mean) / mid_std

# plot z-scores
plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], z_score, label='bid_zscore', lw=1)
plt.axhline(1, linestyle='--', color='red', alpha=0.7)
plt.axhline(-1, linestyle='--', color='green', alpha=0.7)
plt.xlabel('Timestamp')
plt.ylabel('Z-Score')
plt.title('Z-Scores of Mid Price')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# --- 1. Load and Merge Order Book Data ---
ob_files = [".\\r5_d2.csv", ".\\r5_d3.csv", ".\\r5_d4.csv"]
merged_ob_file = ".\\merged_r5.csv"
if not Path(merged_ob_file).exists():
    parts = []
    for i, fn in enumerate(ob_files):
        if Path(fn).exists():
            part = pd.read_csv(fn, delimiter=";")
            part["timestamp"] = part["timestamp"] + i * 1_000_000
            parts.append(part)
    if parts:
        pd.concat(parts, ignore_index=True).to_csv(merged_ob_file, sep=";", index=False)
df_ob = pd.read_csv(merged_ob_file, delimiter=";")

# --- 2. Load and Merge Trades Data ---
tr_files = [".\\tr5_d2.csv", ".\\tr5_d3.csv", ".\\tr5_d4.csv"]
merged_tr_file = ".\\merged_tr5.csv"
if not Path(merged_tr_file).exists():
    parts = []
    for i, fn in enumerate(tr_files):
        if Path(fn).exists():
            part = pd.read_csv(fn, delimiter=";")
            part["timestamp"] = part["timestamp"] + i * 1_000_000
            parts.append(part)
    if parts:
        pd.concat(parts, ignore_index=True).to_csv(merged_tr_file, sep=";", index=False)

if Path(merged_tr_file).exists():
    df_tr = pd.read_csv(merged_tr_file, delimiter=";")
else:
    df_tr = pd.DataFrame()

# --- 3. Configuration ---
TARGET_PRODUCT = 'MICROCHIP_CIRCLE'  # Change to your target product
USE_ROLLING_STATS = True  # Set False to use overall mean/stddev
ROLLING_WINDOW = 30       # Window size if USE_ROLLING_STATS is True

# Set min and max timestamps (None for no limit)
MIN_TIMESTAMP = 2600000  # e.g., 500000
MAX_TIMESTAMP = None  # e.g., 1500000

# --- Vertical Window Lines Configuration ---
SHOW_WINDOW_LINES = True
WINDOW_STEP = ROLLING_WINDOW * 100
# --------------------------------------------

# Filter Data
prod_ob = df_ob[df_ob['product'] == TARGET_PRODUCT].copy()
prod_ob = prod_ob[(prod_ob['bid_price_1'].notna()) & (prod_ob['ask_price_1'].notna())].copy()

if MIN_TIMESTAMP is not None:
    prod_ob = prod_ob[prod_ob['timestamp'] >= MIN_TIMESTAMP]
if MAX_TIMESTAMP is not None:
    prod_ob = prod_ob[prod_ob['timestamp'] <= MAX_TIMESTAMP]

# Note: Trades data typically uses 'symbol' instead of 'product'
if not df_tr.empty and 'symbol' in df_tr.columns:
    prod_tr = df_tr[df_tr['symbol'] == TARGET_PRODUCT].copy()
elif not df_tr.empty and 'product' in df_tr.columns:
    prod_tr = df_tr[df_tr['product'] == TARGET_PRODUCT].copy()
else:
    prod_tr = pd.DataFrame()

if not prod_tr.empty:
    if MIN_TIMESTAMP is not None:
        prod_tr = prod_tr[prod_tr['timestamp'] >= MIN_TIMESTAMP]
    if MAX_TIMESTAMP is not None:
        prod_tr = prod_tr[prod_tr['timestamp'] <= MAX_TIMESTAMP]

# Compute Best Price / Mid Price
prod_ob['best_bid'] = prod_ob[['bid_price_1', 'bid_price_2', 'bid_price_3']].max(axis=1)
prod_ob['best_ask'] = prod_ob[['ask_price_1', 'ask_price_2', 'ask_price_3']].min(axis=1)
prod_ob['mid_price'] = (prod_ob['best_bid'] + prod_ob['best_ask']) / 2

def add_window_lines():
    if SHOW_WINDOW_LINES:
        t_start = prod_ob['timestamp'].min()
        t_end = prod_ob['timestamp'].max()
        # Generate lines based on step
        for t in range(int(t_start), int(t_end), WINDOW_STEP):
            plt.axvline(x=t, color='gray', linestyle='--', alpha=0.3, lw=0.8)

# === Graph 1: All Bid and Ask Prices ===
plt.figure(figsize=(14, 6))
plt.plot(prod_ob['timestamp'], prod_ob['bid_price_1'], label='Bid 1', color='g', lw=0.5)
plt.plot(prod_ob['timestamp'], prod_ob['bid_price_2'], label='Bid 2', color='lightgreen', lw=0.5, alpha=0.7)
plt.plot(prod_ob['timestamp'], prod_ob['bid_price_3'], label='Bid 3', color='palegreen', lw=0.5, alpha=0.5)
plt.plot(prod_ob['timestamp'], prod_ob['ask_price_1'], label='Ask 1', color='r', lw=0.5)
plt.plot(prod_ob['timestamp'], prod_ob['ask_price_2'], label='Ask 2', color='lightcoral', lw=0.5, alpha=0.7)
plt.plot(prod_ob['timestamp'], prod_ob['ask_price_3'], label='Ask 3', color='mistyrose', lw=0.5, alpha=0.5)
add_window_lines()
plt.title(f'{TARGET_PRODUCT} - Bid/Ask Levels')
plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()

# === Graph 2: Best Bid/Ask + Trades Overlay ===
plt.figure(figsize=(14, 6))
plt.plot(prod_ob['timestamp'], prod_ob['best_bid'], label='Best Bid', color='g', lw=0.7)
plt.plot(prod_ob['timestamp'], prod_ob['best_ask'], label='Best Ask', color='r', lw=0.7)

if not prod_tr.empty:
    # Scatter plot for trades. Size can be proportional to quantity
    sizes = prod_tr['quantity'].abs() * 2 
    plt.scatter(prod_tr['timestamp'], prod_tr['price'], 
                c='blue', s=sizes, alpha=0.6, label='Trades', zorder=5)

add_window_lines()
plt.title(f'{TARGET_PRODUCT} - Best Bid/Ask & Traded Volume')
plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()

# === Graph 3: Z-Score Plot ===
if USE_ROLLING_STATS:
    mean_series = prod_ob['mid_price'].rolling(ROLLING_WINDOW).mean()
    std_series = prod_ob['mid_price'].rolling(ROLLING_WINDOW).std()
else:
    mean_series = prod_ob['mid_price'].mean()
    std_series = prod_ob['mid_price'].std()

prod_ob['z_score'] = (prod_ob['mid_price'] - mean_series) / std_series

plt.figure(figsize=(14, 6))
plt.plot(prod_ob['timestamp'], prod_ob['z_score'], label='Mid Price Z-Score', color='purple', lw=1)
plt.axhline(0, color='black', lw=1, linestyle='--')
plt.axhline(2, color='red', lw=1.5, linestyle=':', label='+2 Std Dev')
plt.axhline(-2, color='green', lw=1.5, linestyle=':', label='-2 Std Dev')
add_window_lines()
plt.title(f'{TARGET_PRODUCT} - Z-Score ({"Rolling Window: " + str(ROLLING_WINDOW) if USE_ROLLING_STATS else "Overall Population"})')
plt.xlabel('Timestamp')
plt.ylabel('Z-Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import os

# Create output directory if it doesn't exist
output_dir = "../graphs"
os.makedirs(output_dir, exist_ok=True)

# Define product categories
categories = {
    "galaxy_sounds": [
        "GALAXY_SOUNDS_DARK_MATTER", "GALAXY_SOUNDS_BLACK_HOLES", 
        "GALAXY_SOUNDS_PLANETARY_RINGS", "GALAXY_SOUNDS_SOLAR_WINDS", 
        "GALAXY_SOUNDS_SOLAR_FLAMES"
    ],
    "sleep_pod": [
        "SLEEP_POD_SUEDE", "SLEEP_POD_LAMB_WOOL", "SLEEP_POD_POLYESTER", 
        "SLEEP_POD_NYLON", "SLEEP_POD_COTTON"
    ],
    "microchip": [
        "MICROCHIP_CIRCLE", "MICROCHIP_OVAL", "MICROCHIP_SQUARE", 
        "MICROCHIP_RECTANGLE", "MICROCHIP_TRIANGLE"
    ],
    "pebbles": [
        "PEBBLES_XS", "PEBBLES_S", "PEBBLES_M", "PEBBLES_L", "PEBBLES_XL"
    ],
    "robot": [
        "ROBOT_VACUUMING", "ROBOT_MOPPING", "ROBOT_DISHES", 
        "ROBOT_LAUNDRY", "ROBOT_IRONING"
    ],
    "uv_visor": [
        "UV_VISOR_YELLOW", "UV_VISOR_AMBER", "UV_VISOR_ORANGE", 
        "UV_VISOR_RED", "UV_VISOR_MAGENTA"
    ],
    "translator": [
        "TRANSLATOR_SPACE_GRAY", "TRANSLATOR_ASTRO_BLACK", "TRANSLATOR_ECLIPSE_CHARCOAL", 
        "TRANSLATOR_GRAPHITE_MIST", "TRANSLATOR_VOID_BLUE"
    ],
    "panel": [
        "PANEL_1X2", "PANEL_2X2", "PANEL_1X4", "PANEL_2X4", "PANEL_4X4"
    ],
    "oxygen_shake": [
        "OXYGEN_SHAKE_MORNING_BREATH", "OXYGEN_SHAKE_EVENING_BREATH", 
        "OXYGEN_SHAKE_MINT", "OXYGEN_SHAKE_CHOCOLATE", "OXYGEN_SHAKE_GARLIC"
    ],
    "snackpack": [
        "SNACKPACK_CHOCOLATE", "SNACKPACK_VANILLA", "SNACKPACK_PISTACHIO", 
        "SNACKPACK_STRAWBERRY", "SNACKPACK_RASPBERRY"
    ]
}

# Inverse mapping for easy lookup
product_to_category = {p: cat for cat, products in categories.items() for p in products}

# Iterate through each product
for product in df_ob['product'].unique():
    # Filter order book data for this product
    product_data = df_ob[df_ob['product'] == product].copy()
    product_data = product_data[(product_data['bid_price_1'].notna()) & (product_data['ask_price_1'].notna())]
    
    if len(product_data) == 0:
        continue
    
    # Determine subfolder
    category = product_to_category.get(product, "other")
    cat_dir = os.path.join(output_dir, category)
    os.makedirs(cat_dir, exist_ok=True)
    
    # Create figure
    plt.figure(figsize=(8, 4))
    # plt.plot(product_data['timestamp'], product_data['bid_price_1'], label='Bid 1', lw=0.5)
    # plt.plot(product_data['timestamp'], product_data['bid_price_2'], label='Bid 2', lw=0.5, alpha=0.7)
    # plt.plot(product_data['timestamp'], product_data['bid_price_3'], label='Bid 3', lw=0.5, alpha=0.5)
    plt.plot(product_data['timestamp'], product_data['mid_price'], label='Mid Price', lw=0.7, color='purple')
    # plt.plot(product_data['timestamp'], product_data['ask_price_1'], label='Ask 1', lw=0.5)
    # plt.plot(product_data['timestamp'], product_data['ask_price_2'], label='Ask 2', lw=0.5, alpha=0.7)
    # plt.plot(product_data['timestamp'], product_data['ask_price_3'], label='Ask 3', lw=0.5, alpha=0.5)

    spread = product_data['ask_price_1'] - product_data['bid_price_1']
    mean_spread = spread.mean()
    print(f"Mean spread for {product}: {mean_spread:.2f}")
    
    plt.title(f'{product} - Mid Price')
    plt.xlabel('Timestamp')
    plt.ylabel('Price')
    plt.legend()
    plt.tight_layout()
    
    # Save figure
    file_path = os.path.join(cat_dir, f"{product}.png")
    plt.savefig(file_path, dpi=100)
    plt.close()

print(f"Graphs saved by category to {output_dir}")

In [ ]:
import os

# overlay all products per category and save as overall.png in each category folder

for cat, products in categories.items():
    cat_dir = os.path.join(output_dir, cat)
    os.makedirs(cat_dir, exist_ok=True)

    plt.figure(figsize=(14, 6))
    plotted = False
    for p in products:
        product_data = df_ob[df_ob['product'] == p].copy()
        product_data = product_data[(product_data['bid_price_1'].notna()) & (product_data['ask_price_1'].notna())]
        if product_data.empty:
            continue
        # ensure mid_price exists
        if 'mid_price' not in product_data.columns:
            # product_data['best_bid'] = product_data[['bid_price_1', 'bid_price_2', 'bid_price_3']].max(axis=1)
            # product_data['best_ask'] = product_data[['ask_price_1', 'ask_price_2', 'ask_price_3']].min(axis=1)
            product_data['mid_price'] = (product_data['best_bid'] + product_data['best_ask']) / 2
        plt.plot(product_data['timestamp'], product_data['mid_price'], label=p, lw=0.8, alpha=0.9)
        plotted = True

    if not plotted:
        plt.close()
        continue

    plt.title(f"{cat} - All Products (Mid Price)")
    plt.xlabel("Timestamp")
    plt.ylabel("Mid Price")
    plt.legend(fontsize='small', ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(cat_dir, "overall.png"), dpi=150)
    plt.close()